In [3]:
import pandas as pd
import smogn

df = pd.read_csv('../data/trial_engineered_ds.csv')

# target column
target_col = "CS 28 day (Mpa)"

# generate synthetic samples
df_synthetic = smogn.smoter(
    data=df,
    y=target_col,
    samp_method="balance",   # force oversampling
    k=5
)

# keep only new synthetic rows
synthetic_only = df_synthetic.iloc[len(df):]

# concatenate with original dataframe
df_final = pd.concat([df, synthetic_only], ignore_index=True)

print("Original:", df.shape)
print("Synthetic:", synthetic_only.shape)
print("Final:", df_final.shape)

r_index: 100%|##########| 24/24 [00:00<00:00, 562.88it/s]

Original: (445, 29)
Synthetic: (0, 29)
Final: (445, 29)


In [ ]:
X = df.drop("CS 28 day (Mpa)", axis=1)
y = df['CS 28 day (Mpa)']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Parameter grid
param_grid = {
    "kernel": ["linear", "rbf", "poly"],
    "C": [0.1, 1, 10, 100],
    "epsilon": [0.01, 0.1, 0.5],
    "gamma": ["scale", "auto"]  # used by rbf and poly
}

# Grid Search
svm_model = SVR()
grid_search = GridSearchCV(svm_model, param_grid, cv=5, scoring="r2", n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

# Best model
print("Best Params:", grid_search.best_params_)
best_model = grid_search.best_estimator_

# Predict & Evaluate
y_pred = best_model.predict(X_test)
print("SVM R2:", r2_score(y_test, y_pred))
print("SVM RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))